# 03 — The residual stream and two backward paths

A residual sublayer writes an update into the current state: $Y=X+F(X)$. It is not concatenation, and X is not a fresh copy of the original token embedding at every layer.

For upstream gradient G, backward yields $G_X=G+J_F^\top G$. We will make both terms visible without claiming that cancellation is impossible.

## How to work through this notebook

Run setup once. At each checkpoint, write a prediction and try the small implementation before reading its adjacent reference solution. All reference cells run unchanged from top to bottom; exercise cells contain safe, optional starting points. Numerical checks use CPU float64 unless explicitly noted. Agent-verified reference execution is separate from your learning progress.

In [ ]:
from pathlib import Path
import sys, copy, math, inspect
from dataclasses import replace
import torch
from torch import nn
from torch.nn import functional as F
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/dongxi_llms/decoder_lab.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the Dongxi_LLMs repository")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from dongxi_llms.decoder_lab import (
    DecoderConfig, TinyDecoder, DecoderBlock, MultiHeadAttention, MLP, RMSNorm,
    layer_norm, rms_norm, rope, attend, parameter_count, analytical_parameters,
    cost_estimate, teaching_batch, next_token_loss, fit_one_batch)
torch.set_num_threads(1)
torch.manual_seed(505)
DTYPE = torch.float64
def close(actual, expected, atol=1e-10, rtol=1e-8):
    torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol)
print("CPU reference environment:", torch.__version__)


## 1. Implement a residual update

Use a small linear branch. What happens when its weights are zero?

**Your prediction:** _Write it here before running the reference._

In [ ]:
x = torch.randn(2, 6, 8, dtype=DTYPE, requires_grad=True)
weight = torch.randn(8, 8, dtype=DTYPE) / 4
# Your implementation: y = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
branch = x @ weight
y = x + branch
identity = x + x @ torch.zeros_like(weight)
close(identity, x)
print("State/update/output shapes:", x.shape, branch.shape, y.shape)
print("Update norm:", float(branch.norm().detach()))

### Why this works

The addition preserves the tensor shape. Zero branch output makes the residual sublayer an identity map, whereas a branch used alone would output zero.

## 2. Identify both gradient contributions

For an arbitrary upstream probe G, derive the gradient through X + XW. Compare it with autograd.

**Your prediction:** _Write it here before running the reference._

In [ ]:
probe = torch.randn_like(y)
# Your implementation: manual_gradient = ...

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
manual_gradient = probe + probe @ weight.T
actual_gradient = torch.autograd.grad((y*probe).sum(), x)[0]
close(actual_gradient, manual_gradient)
print("Skip gradient norm:", float(probe.norm()))
print("Branch gradient norm:", float((probe @ weight.T).norm()))
print("Total gradient norm:", float(actual_gradient.norm()))

### Why this works

The same loss sends feedback down both paths, and contributions add at their shared input. Their norms do not generally add: directions matter.

## 3. Test the limits of the identity-path intuition

Could a residual branch erase information or cancel the incoming gradient? Construct F(X) = -X.

**Your prediction:** _Write it here before running the reference._

In [ ]:
# Predict Y and dY/dX before running.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
z = x + (-x)
grad = torch.autograd.grad(z.sum(), x)[0]
close(z, torch.zeros_like(x))
close(grad, torch.zeros_like(x))
print("Output norm and gradient norm:", float(z.norm().detach()), float(grad.norm()))

### Why this works

The direct path exists, but the learned path can oppose it. Residual connections make identity behavior easy to represent; they do not guarantee preservation of information or nonzero gradients.

## 4. Connect the identity case to a full pre-norm block

Zero only the attention and MLP branches in the reusable block. Leave normalization parameters intact.

**Your prediction:** _Write it here before running the reference._

In [ ]:
block = DecoderBlock(DecoderConfig()).double()
states = torch.randn(2, 6, 16, dtype=DTYPE, requires_grad=True)
# Your implementation: zero the branch parameters, not every parameter.

### Reference solution

Run after your attempt; compare the mechanism, not just the final numbers.

In [ ]:
with torch.no_grad():
    for module in (block.attn, block.mlp):
        for parameter in module.parameters():
            parameter.zero_()
output = block(states)[0]
close(output, states)
close(torch.autograd.grad(output.sum(), states)[0], torch.ones_like(states))
print("Full pre-norm block zero-update error:", float((output-states).abs().max().detach()))

### Why this works

Pre-norm leaves the direct stream untouched while normalized states enter the branches. The next notebook tests how post-norm placement changes this identity case.

## Takeaway and evidence boundary

Next: normalization regulates what each branch reads. Keep the residual path separate from the normalized branch input.

Companion map: [Chapter 5 pathway](../day-05/README.md). Reusable source: [decoder_lab.py](../../src/dongxi_llms/decoder_lab.py). Record your explanation and remaining questions here; the notebook's existence does not mark the lesson complete.